# Lesson 2.2 — Business Transaction Performance and Health

## Learning Objectives
- Interpret BT performance metrics: response time, error rate, call rate, slow/very slow counts
- Understand AppDynamics' transaction health classification (Normal, Slow, Very Slow, Stall, Error)
- Identify performance baselines and thresholds
- Analyze BT performance data using the REST API and Python

**Reference**: [Business Transactions](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/business-transactions) | [Transaction Thresholds](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.2.0/business-transactions/monitor-the-business-transaction-performance/transaction-thresholds)

## 1. The Five Health States

Every Business Transaction call is classified into one of five states:

| State | Condition | Action |
|-------|-----------|--------|
| **Normal** | Response time within the slow threshold | All good |
| **Slow** | Response time between slow and very slow threshold | Warning |
| **Very Slow** | Response time exceeds very slow threshold | Critical |
| **Stall** | Response time exceeds the stall threshold (no completion) | Critical |
| **Error** | An exception or HTTP error code occurred | Critical |

**Default thresholds** (configurable per BT):
- Slow: 3× the **learned baseline** (or a static value you set)
- Very Slow: 6× the learned baseline (or static)
- Stall: 2 minutes (no response at all)

This is the key feature of AppDynamics: instead of setting arbitrary static thresholds, the platform **learns what's normal** for each BT and alerts when behavior deviates.

In [ ]:
import random, statistics
from collections import Counter

random.seed(42)

def classify_transaction(response_time_ms: float,
                          baseline_ms: float,
                          is_error: bool = False,
                          stall_threshold_ms: float = 120_000) -> str:
    """Classify a single transaction call into AppDynamics health states."""
    if is_error:
        return "Error"
    if response_time_ms >= stall_threshold_ms:
        return "Stall"
    if response_time_ms >= baseline_ms * 6:
        return "Very Slow"
    if response_time_ms >= baseline_ms * 3:
        return "Slow"
    return "Normal"

# Simulate 200 calls to /checkout endpoint
# Baseline: 300ms (learned normal behavior)
baseline = 300.0
calls = []
for _ in range(190):  # 95% normal calls
    calls.append((random.gauss(baseline, 60), False))
for _ in range(5):    # 2.5% slow calls
    calls.append((random.gauss(baseline * 4, 100), False))
for _ in range(3):    # 1.5% very slow
    calls.append((random.gauss(baseline * 8, 200), False))
for _ in range(2):    # 1% errors
    calls.append((random.gauss(baseline, 30), True))

# Classify all calls
classified = [classify_transaction(rt, baseline, err) for rt, err in calls]
counts = Counter(classified)

print(f"BT Health Report: /checkout ({len(calls)} calls)")
print(f"Baseline: {baseline:.0f} ms")
print(f"Slow threshold (3×): {baseline*3:.0f} ms")
print(f"Very Slow threshold (6×): {baseline*6:.0f} ms")
print()
total = len(calls)
for state in ["Normal", "Slow", "Very Slow", "Stall", "Error"]:
    n = counts.get(state, 0)
    bar = "█" * (n // 2)
    print(f"  {state:<12}: {n:>4} ({n/total*100:.1f}%) {bar}")

## 2. Key BT Performance Metrics

For each BT, AppDynamics tracks these metrics every minute:

| Metric Name (in path) | Meaning |
|-----------------------|---------|
| `Calls per Minute` | How many times this BT was invoked per minute |
| `Average Response Time (ms)` | Mean response time across all calls in that minute |
| `Errors per Minute` | Number of errored calls per minute |
| `Error Rate %` | Errors / Total calls × 100 |
| `Number of Slow Calls` | Count of Slow-state calls |
| `Number of Very Slow Calls` | Count of Very Slow-state calls |
| `Stall Count` | Count of Stall-state calls |

**SLA monitoring tip**: Watch `Number of Very Slow Calls` and `Errors per Minute` — these directly impact user experience.

In [ ]:
# Simulate a 30-minute BT performance time series and visualize it
import matplotlib.pyplot as plt
import numpy as np

random.seed(10)
minutes = list(range(30))

# Normal operation: first 20 mins
# Spike event: minutes 20-25 (deployment gone wrong)
# Recovery: minutes 25-30

avg_rt = []
calls_pm = []
errors_pm = []

for m in minutes:
    if m < 20:
        avg_rt.append(random.gauss(300, 30))
        calls_pm.append(random.gauss(45, 5))
        errors_pm.append(random.gauss(0.5, 0.3))
    elif m < 25:
        avg_rt.append(random.gauss(2800, 400))  # ← spike!
        calls_pm.append(random.gauss(30, 8))
        errors_pm.append(random.gauss(8, 2))
    else:
        avg_rt.append(random.gauss(310, 35))
        calls_pm.append(random.gauss(43, 5))
        errors_pm.append(random.gauss(0.6, 0.3))

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax1.plot(minutes, avg_rt, color='steelblue', linewidth=2)
ax1.axhline(300*3, color='orange', linestyle='--', label='Slow threshold (900ms)')
ax1.axhline(300*6, color='red', linestyle='--', label='Very Slow threshold (1800ms)')
ax1.set_ylabel('Avg Response Time (ms)')
ax1.set_title('/checkout — Business Transaction Performance (30 min)')
ax1.legend(loc='upper right', fontsize=8)
ax1.fill_between(minutes, avg_rt, alpha=0.1, color='steelblue')

ax2.bar(minutes, calls_pm, color='mediumseagreen', alpha=0.7)
ax2.set_ylabel('Calls per Minute')

ax3.bar(minutes, errors_pm, color='tomato', alpha=0.8)
ax3.set_ylabel('Errors per Minute')
ax3.set_xlabel('Minute')

# Highlight the incident window
for ax in [ax1, ax2, ax3]:
    ax.axvspan(20, 25, alpha=0.15, color='red', label='Incident window')

plt.tight_layout()
plt.savefig('bt_performance.png', dpi=100, bbox_inches='tight')
plt.show()
print("Chart saved as bt_performance.png")

## 3. Transaction Snapshots Deep Dive

When a call is classified as Slow, Very Slow, or Error, AppDynamics captures a **Transaction Snapshot** automatically.

### What's in a Transaction Snapshot:
1. **Call Graph** — every Java/Node.js method in the execution path, with time spent
2. **Exit Calls** — calls to databases, HTTP services, message queues
   - SQL statements (with parameters if configured)
   - HTTP call URL and response code
3. **Errors and Exceptions** — stack traces
4. **User Data** — any custom data you've instrumented

### How to view snapshots in the UI:
1. Navigate to **Application → Business Transactions**
2. Click a BT name (e.g., `/checkout`)
3. In the BT detail view, scroll to **Transaction Snapshots**
4. Filter by time, health state, or response time
5. Click a snapshot row to open the call graph

> **Tip**: AppDynamics captures snapshots for slow/error calls and also **continuously samples** a percentage of all calls. Set **Diagnostic Data Collection** to increase sampling during incidents.

In [ ]:
# EXERCISE 2.2: Compute a BT performance summary from raw call data
#
# You receive a list of (response_time_ms, is_error) tuples.
# TASK: Compute the performance summary:
#   - Total calls
#   - Average response time
#   - Error rate %
#   - Counts of Normal / Slow / Very Slow / Error
#
# Use classify_transaction() defined earlier in this notebook.

import random
random.seed(2024)

BASELINE_MS = 250.0

# 1-minute sample of /search BT calls
search_calls = (
    [(random.gauss(250, 40), False) for _ in range(180)] +
    [(random.gauss(900, 100), False) for _ in range(12)] +
    [(random.gauss(2000, 300), False) for _ in range(4)] +
    [(random.gauss(200, 20), True) for _ in range(4)]
)
random.shuffle(search_calls)

def bt_performance_summary(calls: list, baseline_ms: float) -> dict:
    """
    TASK: Compute a BT performance summary dict.
    
    Returns:
    {
        'total_calls': int,
        'avg_response_time': float,
        'error_rate_pct': float,
        'health_counts': {'Normal': int, 'Slow': int, 'Very Slow': int, 'Stall': int, 'Error': int}
    }
    """
    # TODO: Implement using classify_transaction()
    pass

result = bt_performance_summary(search_calls, BASELINE_MS)
if result:
    print("BT Performance Summary: /search")
    print(f"  Total calls       : {result['total_calls']}")
    print(f"  Avg Response Time : {result['avg_response_time']:.1f} ms")
    print(f"  Error Rate        : {result['error_rate_pct']:.2f}%")
    print(f"  Health Breakdown  : {result['health_counts']}")
else:
    print("Implement bt_performance_summary() above to see the result.")

## Summary

- Every BT call is classified: **Normal, Slow, Very Slow, Stall, or Error**
- Thresholds are based on **dynamic baselines** (3× and 6× of learned normal behavior)
- Key metrics: Calls per Minute, Average Response Time, Errors per Minute, Slow/VSlw Counts
- **Transaction Snapshots** capture the full call graph for slow/error calls — your best debugging tool

---

**Next**: [Lesson 2.3 — Flow Maps and Distributed Tracing](lesson-2.3-flow-maps-distributed-tracing.ipynb)